# llamatelemetry SDK v0.1.0 - Kaggle 2× T4 Build Notebook
**Binary Artifact Version:** v0.1.0 (llama.cpp binaries)

## Architecture: Split-GPU Workload

```
┌───────────────────────────┬───────────────────────────────┐
│         GPU 0             │            GPU 1              │
│  llama-server (GGUF)      │  RAPIDS + Graphistry          │
│  LLM Inference            │  Graph Visualization (cuGraph)│
│  15GB VRAM                │  15GB VRAM                    │
└───────────────────────────┴───────────────────────────────┘
```

This notebook builds llamatelemetry binaries for **split-GPU** operation:
- **GPU 0**: llama-server with GGUF model (LLM inference)
- **GPU 1**: RAPIDS/Graphistry with cuDF/cuGraph (graph simulation)
- **SDK Version**: 0.1.0 (Python package for llamatelemetry)
- **Binary Version**: 0.1.0 (llama.cpp build artifacts in tarball)

## Step 1: Verify Kaggle GPU Environment

In [1]:
# Verify we have 2× T4 GPUs
import subprocess
import os

print("="*70)
print("KAGGLE GPU ENVIRONMENT CHECK")
print("="*70)

# Check nvidia-smi
result = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
gpu_lines = [l for l in result.stdout.strip().split("\n") if l.startswith("GPU")]
print(f"\n📊 Detected GPUs: {len(gpu_lines)}")
for line in gpu_lines:
    print(f"   {line}")

# Check CUDA version
print("\n📊 CUDA Version:")
!nvcc --version | grep release

# Check total VRAM
print("\n📊 VRAM Summary:")
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

# Verify we have 2 GPUs
if len(gpu_lines) >= 2:
    print("\n✅ Multi-GPU environment confirmed! Ready for dual-T4 build.")
else:
    print("\n⚠️ WARNING: Less than 2 GPUs detected!")
    print("   Enable 'GPU T4 x2' in Kaggle notebook settings.")

KAGGLE GPU ENVIRONMENT CHECK

📊 Detected GPUs: 2
   GPU 0: Tesla T4 (UUID: GPU-d4bc8786-c526-9099-871f-c027a1330c8b)
   GPU 1: Tesla T4 (UUID: GPU-cf35d9a5-8a82-6f99-f701-0a2549a1c3a8)

📊 CUDA Version:
Cuda compilation tools, release 12.5, V12.5.82

📊 VRAM Summary:
index, name, memory.total [MiB]
0, Tesla T4, 15360 MiB
1, Tesla T4, 15360 MiB

✅ Multi-GPU environment confirmed! Ready for dual-T4 build.


## Step 2: Verify/Install Build Dependencies

**Note:** Kaggle 2× T4 comes with cmake and ninja pre-installed. We only install what's missing.
This step also ensures **NCCL** is available and installs the Python integrations used by
llamatelemetry (Unsloth, Graphistry, OpenTelemetry).


In [2]:
%%time
# Check pre-installed build tools (Kaggle 2× T4 has cmake/ninja)
import os
import subprocess
from pathlib import Path

print("Checking build dependencies...")

# Check CMake
cmake_result = subprocess.run(["cmake", "--version"], capture_output=True, text=True)
if cmake_result.returncode == 0:
    cmake_ver = cmake_result.stdout.split("\n")[0]
    print(f"✅ {cmake_ver}")
else:
    print("⚠️  CMake not found, installing...")
    !apt-get update -qq && apt-get install -y -qq cmake

# Check Ninja
ninja_result = subprocess.run(["ninja", "--version"], capture_output=True, text=True)
if ninja_result.returncode == 0:
    print(f"✅ Ninja {ninja_result.stdout.strip()}")
else:
    print("⚠️  Ninja not found, installing...")
    !apt-get install -y -qq ninja-build

# Check ccache (optional but speeds up rebuilds)
ccache_result = subprocess.run(["which", "ccache"], capture_output=True, text=True)
if ccache_result.returncode != 0:
    print("📦 Installing ccache...")
    !apt-get install -y -qq ccache

# Ensure NCCL runtime + dev libraries are available
print("\n🔌 Checking NCCL libraries...")
nccl_candidates = [
    Path("/usr/lib/x86_64-linux-gnu/libnccl.so"),
    Path("/usr/lib/x86_64-linux-gnu/libnccl.so.2"),
    Path("/usr/local/cuda/lib64/libnccl.so"),
    Path("/usr/local/cuda/targets/x86_64-linux/lib/libnccl.so"),
]
has_nccl = any(p.exists() for p in nccl_candidates)
if not has_nccl:
    print("📦 NCCL not found in system paths, installing via apt...")
    !apt-get update -qq
    !apt-get install -y -qq libnccl2 libnccl-dev || true
    has_nccl = any(p.exists() for p in nccl_candidates)

# Fallback: build NCCL from source if still missing
if not has_nccl:
    print("⚠️  NCCL not found via apt. Building from source (this can take a few minutes)...")
    !rm -rf /kaggle/working/nccl
    !git clone --depth 1 https://github.com/NVIDIA/nccl /kaggle/working/nccl
    %cd /kaggle/working/nccl
    !make -j src.build
    %cd /kaggle/working
    has_nccl = Path("/kaggle/working/nccl/build/lib/libnccl.so").exists()

if has_nccl:
    print("✅ NCCL libraries available")
else:
    print("❌ NCCL build/install failed. You can still build llama.cpp,")
    print("   but NCCL will not be bundled into the final package.")

# Install Python dependencies (including integrations)
print("\n📦 Installing Python integrations...")
py_deps = [
    "huggingface_hub",
    "sseclient-py",
    "unsloth",
    "unsloth_zoo",
    "pygraphistry",
    "opentelemetry-api",
    "opentelemetry-sdk",
    "opentelemetry-exporter-otlp-proto-grpc",
    "opentelemetry-exporter-otlp-proto-http",
    "opentelemetry-instrumentation",
    "opentelemetry-instrumentation-requests",
]
for pkg in py_deps:
    try:
        __import__(pkg.replace('-', '_'))
        print(f"   ✅ {pkg}")
    except Exception:
        print(f"   📦 Installing {pkg}...")
        !pip install -q {pkg}

print("\n✅ Build dependencies ready")
!cmake --version | head -1
!ninja --version


Checking build dependencies...
✅ cmake version 3.31.6
✅ Ninja 1.13.0.git.kitware.jobserver-pipe-1
📦 Installing ccache...
Selecting previously unselected package libhiredis0.14:amd64.
(Reading database ... 129073 files and directories currently installed.)
Preparing to unpack .../libhiredis0.14_0.14.1-2_amd64.deb ...
Unpacking libhiredis0.14:amd64 (0.14.1-2) ...
Selecting previously unselected package ccache.
Preparing to unpack .../ccache_4.5.1-1_amd64.deb ...
Unpacking ccache (4.5.1-1) ...
Setting up libhiredis0.14:amd64 (0.14.1-2) ...
Setting up ccache (4.5.1-1) ...
Updating symlinks in /usr/lib/ccache ...
Processing triggers for libc-bin (2.35-0ubuntu3.8) ...
/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_5.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero_v2.so.0 is not a symbo

## Step 2b: Install cuGraph (GPU 1 Workload) - RAPIDS 25.6.0 Compatible

**Important:** Kaggle has RAPIDS 25.6.0 pre-installed (cudf-cu12, cuml-cu12, pylibraft-cu12, etc.).

**DO NOT** upgrade cuda-python or numba-cuda - this breaks the pre-installed RAPIDS packages!

**Solution:** Install cugraph-cu12==25.6.* to match Kaggle's pre-installed RAPIDS version.

**Note:** Some pip dependency warnings are expected but the packages will work correctly.

In [3]:
%%time
# Install cuGraph matching Kaggle's pre-installed RAPIDS 25.6.0
# CRITICAL: Do NOT upgrade cuda-python or numba-cuda - this breaks RAPIDS!

print("="*70)
print("INSTALLING CUGRAPH FOR GPU 1 (RAPIDS 25.6.0 COMPATIBLE)")
print("="*70)

# Step 1: Check pre-installed RAPIDS versions
import subprocess
print("\n📦 Pre-installed RAPIDS packages on Kaggle:")
for pkg in ["cudf-cu12", "cuml-cu12", "pylibraft-cu12", "cuda-python", "numba-cuda"]:
    result = subprocess.run(["pip", "show", pkg], capture_output=True, text=True)
    if "Version:" in result.stdout:
        version = [l for l in result.stdout.split("\n") if l.startswith("Version:")][0]
        print(f"   {pkg}: {version.split(': ')[1]}")
    else:
        print(f"   {pkg}: NOT INSTALLED")

# Step 2: Install cugraph-cu12 matching RAPIDS 25.6.* (Kaggle's version)
# Using pypi.nvidia.com for RAPIDS packages
print("\n📦 Installing cugraph-cu12==25.6.* (matching Kaggle's RAPIDS)...")
!pip install -q --extra-index-url=https://pypi.nvidia.com "cugraph-cu12==25.6.*"

# Step 3: Install graphistry (minimal, no [ai] extras to avoid conflicts)
print("\n📦 Installing graphistry...")
!pip install -q graphistry

# Step 4: Verify RAPIDS imports work
print("\n📦 Final verification:")
try:
    import cudf
    print(f"   ✅ cuDF: {cudf.__version__}")
except ImportError as e:
    print(f"   ❌ cuDF: {e}")

try:
    import cugraph
    print(f"   ✅ cuGraph: {cugraph.__version__}")
except ImportError as e:
    print(f"   ❌ cuGraph: {e}")
    print("   💡 If cuGraph fails, try: Runtime → Restart runtime, then re-run this cell")

try:
    import graphistry
    print(f"   ✅ Graphistry: {graphistry.__version__}")
except Exception as e:
    print(f"   ⚠️  Graphistry: {e}")

print("\n✅ RAPIDS packages installed! If imports fail, restart runtime and re-run.")

INSTALLING CUGRAPH FOR GPU 1 (RAPIDS 25.6.0 COMPATIBLE)

📦 Pre-installed RAPIDS packages on Kaggle:
   cudf-cu12: 25.6.0
   cuml-cu12: 25.6.0
   pylibraft-cu12: 25.6.0
   cuda-python: 12.6.2.post1
   numba-cuda: 0.11.0

📦 Installing cugraph-cu12==25.6.* (matching Kaggle's RAPIDS)...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 32.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 MB 49.5 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.22.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
datasets 4.3.0 requires pyarrow>=21.0.0, but you have pyarrow 19.0.1 which is incompatible.
google-adk 1.22.1 requires opentelemetry-api<=1.37.0,>=1.37.0, but you have openteleme

## Step 3: Clone llama.cpp (Latest Stable)

In [5]:
%%time
import os

# Set working directory
WORK_DIR = "/kaggle/working"
os.chdir(WORK_DIR)

# Clean any previous build
!rm -rf llama.cpp

# Clone llama.cpp
print("Cloning llama.cpp...")
!git clone --depth 1 https://github.com/ggml-org/llama.cpp.git

os.chdir("llama.cpp")

# Get commit info
print("\n📦 llama.cpp Version:")
!git log -1 --oneline
!git describe --tags --always 2>/dev/null || echo "(no tag)"

Cloning llama.cpp...
Cloning into 'llama.cpp'...
remote: Enumerating objects: 2472, done.
remote: Counting objects: 100% (2472/2472), done.
remote: Compressing objects: 100% (1943/1943), done.
remote: Total 2472 (delta 506), reused 1765 (delta 456), pack-reused 0 (from 0)
Receiving objects: 100% (2472/2472), 27.39 MiB | 19.92 MiB/s, done.
Resolving deltas: 100% (506/506), done.

📦 llama.cpp Version:
9f682fb (grafted, HEAD -> master, origin/master, origin/HEAD) ggml-cpu: FA split across kv for faster TG (#19209)
9f682fb
CPU times: user 62.3 ms, sys: 74.7 ms, total: 137 ms
Wall time: 4.05 s


## Step 4: Configure CMake with CUDA (stub + CMakeLists patch + VMM disabled)

**Critical:** Kaggle has no `libcuda.so` and `/usr/local/cuda/` is read-only. We need three fixes:

1. **Stub library** — Create a minimal `libcuda.so` in `/kaggle/working/cuda_stubs/` so the linker can resolve CUDA driver symbols at build time (the real driver is used at runtime).
2. **CMakeLists patch** — CMake's `FindCUDAToolkit` fails to create the `CUDA::cuda_driver` imported target because it can't find `libcuda.so` in system paths. We inject a target definition at the top of `ggml/src/ggml-cuda/CMakeLists.txt` that points directly at our stub. Without this patch CMake errors at the Generate step with *"Target `CUDA::cuda_driver` links to … but the target was not found"*.
3. **VMM disabled** — `-DGGML_CUDA_NO_VMM` prevents usage of CUDA Virtual Memory Management APIs that require real driver symbols at build time.

**NCCL note:** We will bundle `libnccl.so` into the final package so multi-GPU workflows can use NCCL collectives when available.


In [6]:
%%time
import os
os.chdir("/kaggle/working/llama.cpp")

# Clean previous build
!rm -rf build

print("="*70)
print("STEP 4: CREATE CUDA DRIVER STUB + PATCH CMAKE + CONFIGURE (VMM DISABLED)")
print("="*70)

# ============================================================================
# PART A: Create libcuda.so stub in WRITABLE location
# Kaggle's /usr/local/cuda is read-only, so we use /kaggle/working/
# ============================================================================
print("\n🔧 [A] Creating CUDA driver stub library...")

STUBS_DIR = "/kaggle/working/cuda_stubs"
os.makedirs(STUBS_DIR, exist_ok=True)

# Minimal C stub — provides zero-valued symbol pointers.
# At LINK time this satisfies the linker; at RUNTIME the real CUDA driver
# (injected by the GPU driver into every CUDA process) provides real implementations.
# We disable VMM via -DGGML_CUDA_NO_VMM so we don't need the advanced
# memory management APIs (cuMemCreate, cuMemMap, etc.)
stub_code = '''
// Minimal CUDA driver stub for linking purposes only.
// At runtime, the real driver is used automatically by the GPU driver.

void* cuGetErrorString = 0;
void* cuGetErrorName = 0;
void* cuInit = 0;
void* cuDriverGetVersion = 0;
void* cuDeviceGet = 0;
void* cuDeviceGetCount = 0;
void* cuDeviceGetName = 0;
void* cuDeviceGetAttribute = 0;
void* cuDeviceTotalMem = 0;
void* cuDeviceGetUuid = 0;
void* cuCtxCreate = 0;
void* cuCtxDestroy = 0;
void* cuCtxGetCurrent = 0;
void* cuCtxSetCurrent = 0;
void* cuCtxPushCurrent = 0;
void* cuCtxPopCurrent = 0;
void* cuCtxSynchronize = 0;
void* cuMemAlloc = 0;
void* cuMemFree = 0;
void* cuMemcpy = 0;
void* cuMemcpyHtoD = 0;
void* cuMemcpyDtoH = 0;
void* cuMemcpyDtoD = 0;
void* cuMemsetD8 = 0;
void* cuMemsetD32 = 0;
void* cuModuleLoad = 0;
void* cuModuleUnload = 0;
void* cuModuleGetFunction = 0;
void* cuLaunchKernel = 0;
void* cuStreamCreate = 0;
void* cuStreamDestroy = 0;
void* cuStreamSynchronize = 0;
void* cuEventCreate = 0;
void* cuEventDestroy = 0;
void* cuEventRecord = 0;
void* cuEventSynchronize = 0;
void* cuEventElapsedTime = 0;
'''

stub_c_path = f"{STUBS_DIR}/cuda_stub.c"
with open(stub_c_path, "w") as f:
    f.write(stub_code)

stub_so_path = f"{STUBS_DIR}/libcuda.so"
!gcc -shared -fPIC -o {stub_so_path} {stub_c_path}
!ln -sf {stub_so_path} {STUBS_DIR}/libcuda.so.1

if os.path.exists(stub_so_path):
    size = os.path.getsize(stub_so_path)
    print(f"   ✅ Created libcuda.so stub ({size} bytes) in {STUBS_DIR}")
    !ls -la {STUBS_DIR}
else:
    print("   ❌ Failed to create stub!")

# ============================================================================
# PART B: Patch ggml/src/ggml-cuda/CMakeLists.txt
#
# WHY THIS PATCH IS NEEDED:
#   CMake's FindCUDAToolkit module creates the CUDA::cuda_driver IMPORTED target
#   by searching for libcuda.so in hard-coded system paths:
#       /usr/lib, /usr/local/cuda/lib64/stubs, etc.
#   On Kaggle, /usr/local/cuda is READ-ONLY and no libcuda.so exists in any
#   searchable path.  FindCUDAToolkit therefore never creates the target.
#
#   ggml-cuda's CMakeLists.txt unconditionally does:
#       target_link_libraries(ggml-cuda PRIVATE CUDA::cuda_driver)
#   Since the target doesn't exist, CMake fails at the GENERATE step (after
#   the configure step succeeds) with:
#       "Target 'ggml-cuda' links to: CUDA::cuda_driver but the target was not found."
#
#   CMAKE_LIBRARY_PATH and DCUDAToolkit_LIBRARY_DIR do NOT help because
#   FindCUDAToolkit uses its own internal search logic, not CMAKE_LIBRARY_PATH.
#
# THE FIX:
#   Inject an if(NOT TARGET CUDA::cuda_driver) block at the top of the file
#   that manually creates the target as an INTERFACE library pointing at our stub.
#   This runs BEFORE ggml-cuda tries to link against it.
# ============================================================================
print("\n🔧 [B] Patching ggml-cuda CMakeLists.txt (CUDA::cuda_driver target fix)...")

cmake_lists_path = "ggml/src/ggml-cuda/CMakeLists.txt"
with open(cmake_lists_path, "r") as f:
    cmake_content = f.read()

patch_block = f'''
# ----- llamatelemetry Kaggle patch: define CUDA::cuda_driver stub target -----
# FindCUDAToolkit cannot locate libcuda.so on Kaggle (read-only /usr/local/cuda)
# so it never creates the CUDA::cuda_driver imported target.  We define it here
# as an INTERFACE library that links our stub .so.  The guard ensures we don't
# override it if FindCUDAToolkit succeeds on a normal system.
if(NOT TARGET CUDA::cuda_driver)
    add_library(CUDA::cuda_driver INTERFACE IMPORTED)
    set_target_properties(CUDA::cuda_driver PROPERTIES
        INTERFACE_LINK_LIBRARIES "{STUBS_DIR}/libcuda.so"
    )
endif()
# ----- end llamatelemetry patch -----

'''

# Insert the patch before the first non-comment, non-blank line
lines = cmake_content.split('\n')
insert_idx = 0
for i, line in enumerate(lines):
    stripped = line.strip()
    if stripped and not stripped.startswith('#'):
        insert_idx = i
        break

patched_content = '\n'.join(lines[:insert_idx] + patch_block.split('\n') + lines[insert_idx:])

with open(cmake_lists_path, "w") as f:
    f.write(patched_content)

# Verify
with open(cmake_lists_path, "r") as f:
    verify = f.read()
if "llamatelemetry Kaggle patch" in verify and STUBS_DIR in verify:
    print(f"   ✅ Patched {cmake_lists_path}")
    print(f"       → CUDA::cuda_driver will resolve to {STUBS_DIR}/libcuda.so")
else:
    print(f"   ❌ Patch verification failed — check {cmake_lists_path} manually")

# ============================================================================
# PART C: Set link-time environment + run CMake
# ============================================================================

# LIBRARY_PATH is used by the linker at COMPILE time only.
# CRITICAL: Do NOT set LD_LIBRARY_PATH — that would affect RUNTIME and cause
# the built binaries to load our fake stub instead of the real CUDA driver!
os.environ["LIBRARY_PATH"] = f"{STUBS_DIR}:" + os.environ.get("LIBRARY_PATH", "")

print("\n📦 CMake Configuration:")
print("   Target: SM 7.5 (Tesla T4)")
print("   FlashAttention: All quantization types")
print("   CUDA VMM: DISABLED (-DGGML_CUDA_NO_VMM)")
print("   Static linking: Enabled")
print(f"   CUDA stub path: {STUBS_DIR}")
print("")

cmake_cmd = f"""
cmake -B build -G Ninja \
    -DGGML_CUDA=ON \
    -DCMAKE_CUDA_ARCHITECTURES="75" \
    -DGGML_CUDA_FA_ALL_QUANTS=ON \
    -DGGML_NATIVE=OFF \
    -DBUILD_SHARED_LIBS=OFF \
    -DLLAMA_BUILD_EXAMPLES=ON \
    -DLLAMA_BUILD_TESTS=OFF \
    -DLLAMA_BUILD_SERVER=ON \
    -DCMAKE_BUILD_TYPE=Release \
    -DCMAKE_C_COMPILER=gcc \
    -DCMAKE_CXX_COMPILER=g++ \
    -DCMAKE_C_FLAGS="-DGGML_CUDA_NO_VMM" \
    -DCMAKE_CXX_FLAGS="-DGGML_CUDA_NO_VMM" \
    -DCMAKE_CUDA_FLAGS="-DGGML_CUDA_NO_VMM" \
    -DCMAKE_LIBRARY_PATH="{STUBS_DIR}" \
    -DCUDAToolkit_LIBRARY_DIR="{STUBS_DIR}"
"""

!{cmake_cmd}

# Verify: build.ninja is only written if BOTH Configure AND Generate succeed.
# The previous version of this notebook only checked for build.ninja existence
# after running cmake, which masked the CUDA::cuda_driver Generate-step error
# because build.ninja was never created but the cell still printed success.
if os.path.exists("build/build.ninja"):
    print("\n✅ CMake configuration complete! build.ninja generated successfully.")
else:
    print("\n❌ CMake configuration FAILED — build.ninja was not generated.")
    print("   Check the output above for errors (look for 'CMake Error').")

STEP 4: CREATE CUDA DRIVER STUB + PATCH CMAKE + CONFIGURE (VMM DISABLED)

🔧 [A] Creating CUDA driver stub library...
   ✅ Created libcuda.so stub (16424 bytes) in /kaggle/working/cuda_stubs
total 32
drwxr-xr-x 2 root root  4096 Feb  2 19:01 .
drwxr-xr-x 5 root root  4096 Feb  2 19:01 ..
-rw-r--r-- 1 root root  1087 Feb  2 19:01 cuda_stub.c
-rwxr-xr-x 1 root root 16424 Feb  2 19:01 libcuda.so
lrwxrwxrwx 1 root root    37 Feb  2 19:01 libcuda.so.1 -> /kaggle/working/cuda_stubs/libcuda.so

🔧 [B] Patching ggml-cuda CMakeLists.txt (CUDA::cuda_driver target fix)...
   ✅ Patched ggml/src/ggml-cuda/CMakeLists.txt
       → CUDA::cuda_driver will resolve to /kaggle/working/cuda_stubs/libcuda.so

📦 CMake Configuration:
   Target: SM 7.5 (Tesla T4)
   FlashAttention: All quantization types
   CUDA VMM: DISABLED (-DGGML_CUDA_NO_VMM)
   Static linking: Enabled
   CUDA stub path: /kaggle/working/cuda_stubs

-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.

## Step 5: Build llama.cpp (This takes ~8-12 minutes)

In [7]:
%%time
import os
import multiprocessing
import sys

os.chdir("/kaggle/working/llama.cpp")

# Get CPU count for parallel build
cpu_count = multiprocessing.cpu_count()
print(f"Building with {cpu_count} parallel jobs...")
print("This will take approximately 8-12 minutes.\n")

# Build
build_result = os.system(f"cmake --build build --config Release -j{cpu_count}")

print("\n" + "="*60)

# Verify build succeeded
if build_result == 0 and os.path.exists("build/bin/llama-server"):
    print("✅ BUILD COMPLETE!")
    print("="*60)
    !ls -lh build/bin/llama-server
else:
    print("❌ BUILD FAILED!")
    print("="*60)
    print("Check the build output above for errors.")
    sys.exit(1)

Building with 4 parallel jobs...
This will take approximately 8-12 minutes.

[1/477] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/ggml.cpp.o
[2/477] Building C object ggml/src/CMakeFiles/ggml-base.dir/ggml-alloc.c.o
[3/477] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/ggml-threading.cpp.o
[4/477] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/ggml-backend.cpp.o
[5/477] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/ggml-opt.cpp.o
[6/477] Building C object ggml/src/CMakeFiles/ggml-base.dir/ggml.c.o
[7/477] Building C object ggml/src/CMakeFiles/ggml-cpu.dir/ggml-cpu/ggml-cpu.c.o
[8/477] Building CXX object ggml/src/CMakeFiles/ggml-cpu.dir/ggml-cpu/ggml-cpu.cpp.o
[9/477] Building CXX object ggml/src/CMakeFiles/ggml-cpu.dir/ggml-cpu/hbm.cpp.o
[10/477] Building C object ggml/src/CMakeFiles/ggml-cpu.dir/ggml-cpu/quants.c.o
[11/477] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/gguf.cpp.o
[12/477] Building CXX object ggml/src/CMakeFiles/ggml-cpu.dir/ggm

## Step 6: Verify Built Binaries

In [8]:
import os
os.chdir("/kaggle/working/llama.cpp/build/bin")

print("Built binaries:")
print("="*60)
!ls -lh llama-* 2>/dev/null | head -20

print("\nKey binary sizes:")
!du -h llama-server llama-cli llama-quantize 2>/dev/null

print("\nChecking CUDA support in llama-server:")
!./llama-server --help 2>&1 | grep -i "cuda\|gpu\|ngl" | head -5

Built binaries:
-rwxr-xr-x 1 root root 238M Feb  2 19:26 llama-batched
-rwxr-xr-x 1 root root 238M Feb  2 19:27 llama-batched-bench
-rwxr-xr-x 1 root root 234M Feb  2 19:27 llama-bench
-rwxr-xr-x 1 root root 240M Feb  2 19:27 llama-cli
-rwxr-xr-x 1 root root 238M Feb  2 19:27 llama-completion
-rwxr-xr-x 1 root root 234M Feb  2 19:27 llama-convert-llama2c-to-ggml
-rwxr-xr-x 1 root root 238M Feb  2 19:27 llama-cvector-generator
-rwxr-xr-x 1 root root 238M Feb  2 19:26 llama-debug
-rwxr-xr-x 1 root root 238M Feb  2 19:27 llama-diffusion-cli
-rwxr-xr-x 1 root root 238M Feb  2 19:26 llama-embedding
-rwxr-xr-x 1 root root 238M Feb  2 19:26 llama-eval-callback
-rwxr-xr-x 1 root root 238M Feb  2 19:27 llama-export-lora
-rwxr-xr-x 1 root root 238M Feb  2 19:27 llama-finetune
-rwxr-xr-x 1 root root 238M Feb  2 19:27 llama-fit-params
-rwxr-xr-x 1 root root  17K Feb  2 19:27 llama-gemma3-cli
-rwxr-xr-x 1 root root 238M Feb  2 19:27 llama-gen-docs
-rwxr-xr-x 1 root root 683K Feb  2 19:26 llama-gguf

## Step 7: Test Multi-GPU Support

In [9]:
import os
os.chdir("/kaggle/working/llama.cpp/build/bin")

print("Testing multi-GPU CLI flags:")
print("="*60)

# Check for multi-GPU flags
print("\n📌 --tensor-split (VRAM distribution):")
!./llama-server --help 2>&1 | grep -A2 "tensor-split"

print("\n📌 --split-mode (layer/row splitting):")
!./llama-server --help 2>&1 | grep -A2 "split-mode"

print("\n📌 --main-gpu (primary GPU selection):")
!./llama-server --help 2>&1 | grep -A2 "main-gpu"

print("\n✅ Multi-GPU support confirmed!")

Testing multi-GPU CLI flags:

📌 --tensor-split (VRAM distribution):
-ts,   --tensor-split N0,N1,N2,...      fraction of the model to offload to each GPU, comma-separated list of
                                        proportions, e.g. 3,1
                                        (env: LLAMA_ARG_TENSOR_SPLIT)

📌 --split-mode (layer/row splitting):
-sm,   --split-mode {none,layer,row}    how to split the model across multiple GPUs, one of:
                                        - none: use one GPU only
                                        - layer (default): split layers and KV across GPUs
--
-mg,   --main-gpu INDEX                 the GPU to use for the model (with split-mode = none), or for
                                        intermediate results and KV (with split-mode = row) (default: 0)
                                        (env: LLAMA_ARG_MAIN_GPU)
-fit,  --fit [on|off]                   whether to adjust unset arguments to fit in device memory ('on' or

📌 --main-gpu (prim

## Step 7b: Feature Coverage Notes (llama.cpp + NCCL)

This build includes the full **llama.cpp toolchain** (llama-server, llama-cli,
llama-quantize, GGUF tooling) and bundles **NCCL** for multi-GPU collectives.

Reference documentation:
- llama.cpp repository + server docs: https://github.com/ggml-org/llama.cpp
- llama.cpp server tool: https://github.com/ggml-org/llama.cpp/tree/master/tools/server
- NCCL user guide: https://docs.nvidia.com/deeplearning/nccl/user-guide/docs/index.html
- NCCL source: https://github.com/NVIDIA/nccl
- GGUF format: https://github.com/ggml-org/ggml/blob/master/docs/gguf.md

Verified server capabilities (see `llama-server --help` above):
- OpenAI-compatible chat completions: `/v1/chat/completions`
- Embeddings: `/embedding`
- Reranking: `/reranking`
- Multi-user / parallel decoding (`-np`)
- Speculative decoding (`-md`)
- Grammar-constrained decoding (`--grammar-file`)

NCCL provides collectives like AllReduce, Broadcast, Reduce, AllGather,
ReduceScatter, and Send/Recv for multi-GPU communication.


## Step 8: Create llamatelemetry v0.1.0 Package

This packaging step includes **all llama.cpp tools** plus **CUDA + NCCL libraries**
so the resulting tarball is self-contained for multi-GPU inference.


In [10]:
import os
import shutil
import json
import subprocess
from datetime import datetime

os.chdir("/kaggle/working")

# Package info
VERSION = "0.1.0"
BUILD_DATE = datetime.now().strftime("%Y%m%d")
PACKAGE_NAME = f"llamatelemetry-v{VERSION}-cuda12-kaggle-t4x2"
PACKAGE_DIR = f"/kaggle/working/{PACKAGE_NAME}"

print(f"Creating package: {PACKAGE_NAME}")
print("="*60)

# Create directory structure
os.makedirs(f"{PACKAGE_DIR}/bin", exist_ok=True)
os.makedirs(f"{PACKAGE_DIR}/lib", exist_ok=True)
os.makedirs(f"{PACKAGE_DIR}/include", exist_ok=True)


# Copy NCCL libraries if available
nccl_lib_candidates = [
    "/usr/lib/x86_64-linux-gnu/libnccl.so",
    "/usr/lib/x86_64-linux-gnu/libnccl.so.2",
    "/usr/local/cuda/lib64/libnccl.so",
    "/usr/local/cuda/targets/x86_64-linux/lib/libnccl.so",
    "/kaggle/working/nccl/build/lib/libnccl.so",
]
copied_nccl = []
for nccl_path in nccl_lib_candidates:
    if os.path.exists(nccl_path):
        dest = os.path.join(PACKAGE_DIR, "lib", os.path.basename(nccl_path))
        shutil.copy2(nccl_path, dest)
        copied_nccl.append(dest)

if copied_nccl:
    print("✅ NCCL libraries bundled:")
    for p in copied_nccl:
        print(f"   - {p}")
else:
    print("⚠️  NCCL libraries not found; package will not include libnccl.so")
# Binaries to include
BUILD_BIN = "/kaggle/working/llama.cpp/build/bin"
binaries = [
    # Core server
    "llama-server",
    "llama-cli",
    # Quantization & conversion
    "llama-quantize",
    "llama-gguf",
    "llama-gguf-hash",
    "llama-gguf-split",
    "llama-imatrix",
    # LoRA & embedding
    "llama-export-lora",
    "llama-embedding",
    # Utilities
    "llama-tokenize",
    "llama-infill",
    "llama-perplexity",
    "llama-bench",
    "llama-cvector-generator",
]

# Copy binaries
copied = []
for binary in binaries:
    src = f"{BUILD_BIN}/{binary}"
    if os.path.exists(src):
        shutil.copy2(src, f"{PACKAGE_DIR}/bin/{binary}")
        os.chmod(f"{PACKAGE_DIR}/bin/{binary}", 0o755)
        copied.append(binary)
        print(f"  ✅ {binary}")
    else:
        print(f"  ⚠️  {binary} (not found)")

print(f"\n📦 Copied {len(copied)}/{len(binaries)} binaries")

Creating package: llamatelemetry-v0.1.0-cuda12-kaggle-t4x2
✅ NCCL libraries bundled:
   - /kaggle/working/llamatelemetry-v0.1.0-cuda12-kaggle-t4x2/lib/libnccl.so
   - /kaggle/working/llamatelemetry-v0.1.0-cuda12-kaggle-t4x2/lib/libnccl.so.2
  ✅ llama-server
  ✅ llama-cli
  ✅ llama-quantize
  ✅ llama-gguf
  ✅ llama-gguf-hash
  ✅ llama-gguf-split
  ✅ llama-imatrix
  ✅ llama-export-lora
  ✅ llama-embedding
  ✅ llama-tokenize
  ⚠️  llama-infill (not found)
  ✅ llama-perplexity
  ✅ llama-bench
  ✅ llama-cvector-generator

📦 Copied 13/14 binaries


## Step 9: Create Package Metadata

In [11]:
import json
import subprocess
from datetime import datetime
from pathlib import Path

# Get llama.cpp info
os.chdir("/kaggle/working/llama.cpp")
commit_hash = subprocess.getoutput("git rev-parse HEAD")
commit_date = subprocess.getoutput("git log -1 --format=%ci")
commit_msg = subprocess.getoutput("git log -1 --format=%s")

# Get CUDA version
cuda_version = subprocess.getoutput("nvcc --version | grep release | sed 's/.*release //' | cut -d, -f1")

# NCCL version detection (best-effort)
nccl_version = None
nccl_header = Path("/usr/include/nccl.h")
if nccl_header.exists():
    for line in nccl_header.read_text().splitlines():
        if line.startswith("#define NCCL_VERSION_CODE"):
            nccl_version = line.split()[-1]
            break

# Create metadata
metadata = {
    "package": "llamatelemetry",
    "version": VERSION,
    "build_date": datetime.now().isoformat(),
    "platform": {
        "name": "kaggle",
        "gpu_count": 2,
        "gpu_model": "Tesla T4",
        "vram_per_gpu_gb": 15,
        "total_vram_gb": 30,
        "compute_capability": "7.5",
        "architecture": "Turing"
    },
    "cuda": {
        "version": cuda_version,
        "architectures": ["sm_75"],
        "flash_attention": True,
        "flash_attention_all_quants": True
    },
    "nccl": {
        "version_code": nccl_version,
        "bundled": True,
        "collectives": ["allreduce", "allgather", "reduce", "broadcast", "reducescatter", "sendrecv"]
    },
    "llama_cpp": {
        "commit": commit_hash,
        "commit_date": commit_date,
        "commit_message": commit_msg,
        "server": True,
        "tools": True,
        "openai_compatible": True
    },
}

with open("/kaggle/working/metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("✅ metadata.json created")


✅ metadata.json created


## Step 10: Create README and Usage Guide

In [12]:
readme_content = f'''# llamatelemetry v{VERSION} - Kaggle 2× Tesla T4 Build

Pre-built CUDA 12 binaries for **Kaggle dual Tesla T4** multi-GPU inference with
**OpenTelemetry observability** and **NCCL** bundled.

## Objective

llamatelemetry is a **CUDA-first OpenTelemetry Python SDK for LLM inference
observability and explainability**.  It pairs llama.cpp binaries (multi-GPU
GGUF inference) with an OTel tracing + metrics layer so every inference request
is automatically instrumented — latency, token counts, GPU memory, and NCCL
collective activity can be exported as spans and metrics.

## ✅ Included C++ Features (llama.cpp)

- `llama-server` (OpenAI-compatible HTTP server)
- Chat completions: `/v1/chat/completions`
- Embeddings: `/embedding`
- Reranking: `/reranking`
- Speculative decoding (`-md draft.gguf`)
- Grammar-constrained decoding (`--grammar-file`)
- Multi-user / parallel decoding (`-np`)

## ✅ Included GPU Communication (NCCL)

NCCL provides optimized collectives for multi-GPU inference:
AllReduce, Broadcast, Reduce, AllGather, ReduceScatter, Send/Recv.

## ✅ Python Integrations

- Unsloth: fine-tune + export to GGUF
- Graphistry: GPU-accelerated graph visualization
- OpenTelemetry: tracing + metrics + OTLP exporters
- OpenTelemetry Python Contrib: instrumentation packages

## 🎯 Architecture

```
┌─────────────┐    ┌───────────────────┐    ┌─────────────────┐
│   UNSLOTH   │───▶│  llamatelemetry   │───▶│  llama-server   │
│  Training   │    │  GGUF Conv + OTel │    │  Multi-GPU Inf  │
│  Fine-tune  │    │  NCCL + Graphistry│    │  2× T4 (30GB)   │
└─────────────┘    └───────────────────┘    └─────────────────┘
                                              │
                                              ▼
                                         Graphistry
```

## Binaries Included
- llama-server, llama-cli, llama-quantize, llama-gguf, llama-imatrix, etc.

## NCCL Libraries
- libnccl.so (bundled in /lib if available)

## Usage

```
./start-server.sh model.gguf 8080
```

## Notes
- Ensure LD_LIBRARY_PATH includes ./lib for NCCL runtime
- This build targets Tesla T4 (SM 7.5)
'''


## Step 11: Create Helper Scripts

In [13]:
# Create start-server.sh helper script
start_script = '''#!/bin/bash
# llamatelemetry v0.1.0 - Start Multi-GPU Server
# Usage: ./start-server.sh <model.gguf> [port]

MODEL="$1"
PORT="${2:-8080}"

if [ -z "$MODEL" ]; then
    echo "Usage: $0 <model.gguf> [port]"
    echo "Example: $0 qwen2.5-7b-Q4_K_M.gguf 8080"
    exit 1
fi

SCRIPT_DIR="$(cd "$(dirname "$0")" && pwd)"

echo "Starting llama-server with dual T4 config..."
echo "Model: $MODEL"
echo "Port: $PORT"
echo ""

"$SCRIPT_DIR/bin/llama-server" \\
    --model "$MODEL" \\
    --n-gpu-layers 99 \\
    --tensor-split 0.5,0.5 \\
    --split-mode layer \\
    --flash-attn \\
    --host 0.0.0.0 \\
    --port "$PORT" \\
    --ctx-size 8192 \\
    --batch-size 2048 \\
    --ubatch-size 512 \\
    --parallel 4
'''

with open(f"{PACKAGE_DIR}/start-server.sh", "w") as f:
    f.write(start_script)
os.chmod(f"{PACKAGE_DIR}/start-server.sh", 0o755)

# Create quantize.sh helper script
quantize_script = '''#!/bin/bash
# llamatelemetry v0.1.0 - Quantize Model
# Usage: ./quantize.sh <input.gguf> <output.gguf> [quant_type]

INPUT="$1"
OUTPUT="$2"
QUANT="${3:-Q4_K_M}"

if [ -z "$INPUT" ] || [ -z "$OUTPUT" ]; then
    echo "Usage: $0 <input.gguf> <output.gguf> [quant_type]"
    echo "Quant types: Q4_K_M (default), Q8_0, Q5_K_M, IQ4_XS, etc."
    exit 1
fi

SCRIPT_DIR="$(cd "$(dirname "$0")" && pwd)"

echo "Quantizing: $INPUT → $OUTPUT ($QUANT)"
"$SCRIPT_DIR/bin/llama-quantize" "$INPUT" "$OUTPUT" "$QUANT"
'''

with open(f"{PACKAGE_DIR}/quantize.sh", "w") as f:
    f.write(quantize_script)
os.chmod(f"{PACKAGE_DIR}/quantize.sh", 0o755)

print("✅ Helper scripts created:")
print("   - start-server.sh")
print("   - quantize.sh")

✅ Helper scripts created:
   - start-server.sh
   - quantize.sh


## Step 11b: Verify NCCL Runtime Linking

This step confirms whether the packaged `llama-server` is linked against NCCL.
Even if the binary is not linked directly, we still bundle `libnccl.so` for runtime availability.


In [15]:
# Verify NCCL linkage in llama-server
import os
import subprocess

llama_server = f"{PACKAGE_DIR}/bin/llama-server"
if os.path.exists(llama_server):
    print("🔍 ldd | grep nccl (llama-server)")
    result = subprocess.getoutput(f"ldd {llama_server} | grep -i nccl")
    if result.strip():
        print(result)
        print("✅ NCCL linkage detected")
    else:
        print("⚠️  NCCL linkage not detected (still bundled if libnccl.so copied)")
else:
    print("⚠️  llama-server not found in package path")


🔍 ldd | grep nccl (llama-server)
⚠️  NCCL linkage not detected (still bundled if libnccl.so copied)


## Step 12: Create Distribution Archive

In [16]:
import os
import hashlib

os.chdir("/kaggle/working")

TARBALL = f"{PACKAGE_NAME}.tar.gz"

print(f"Creating distribution archive: {TARBALL}")
print("="*60)

# Create tarball
!tar -czvf {TARBALL} {PACKAGE_NAME}

# Calculate SHA256
with open(TARBALL, "rb") as f:
    sha256 = hashlib.sha256(f.read()).hexdigest()

# Write checksum file
with open(f"{TARBALL}.sha256", "w") as f:
    f.write(f"{sha256}  {TARBALL}\n")

print("\n" + "="*60)
print("📦 DISTRIBUTION PACKAGE READY")
print("="*60)
!ls -lh {TARBALL}*
print(f"\nSHA256: {sha256}")

Creating distribution archive: llamatelemetry-v0.1.0-cuda12-kaggle-t4x2.tar.gz
llamatelemetry-v0.1.0-cuda12-kaggle-t4x2/
llamatelemetry-v0.1.0-cuda12-kaggle-t4x2/start-server.sh
llamatelemetry-v0.1.0-cuda12-kaggle-t4x2/lib/
llamatelemetry-v0.1.0-cuda12-kaggle-t4x2/lib/libnccl.so
llamatelemetry-v0.1.0-cuda12-kaggle-t4x2/lib/libnccl.so.2
llamatelemetry-v0.1.0-cuda12-kaggle-t4x2/quantize.sh
llamatelemetry-v0.1.0-cuda12-kaggle-t4x2/bin/
llamatelemetry-v0.1.0-cuda12-kaggle-t4x2/bin/llama-embedding
llamatelemetry-v0.1.0-cuda12-kaggle-t4x2/bin/llama-perplexity
llamatelemetry-v0.1.0-cuda12-kaggle-t4x2/bin/llama-gguf
llamatelemetry-v0.1.0-cuda12-kaggle-t4x2/bin/llama-tokenize
llamatelemetry-v0.1.0-cuda12-kaggle-t4x2/bin/llama-server
llamatelemetry-v0.1.0-cuda12-kaggle-t4x2/bin/llama-cvector-generator
llamatelemetry-v0.1.0-cuda12-kaggle-t4x2/bin/llama-gguf-hash
llamatelemetry-v0.1.0-cuda12-kaggle-t4x2/bin/llama-bench
llamatelemetry-v0.1.0-cuda12-kaggle-t4x2/bin/llama-imatrix
llamatelemetry-v0.1.

## Step 13: Test Multi-GPU Inference (Optional)

In [17]:
# Download a small test model and verify multi-GPU works
from huggingface_hub import hf_hub_download
import subprocess
import time
import requests
import os
import select

print("Downloading small test model...")
model_path = hf_hub_download(
    repo_id="lmstudio-community/gemma-2-2b-it-GGUF",
    filename="gemma-2-2b-it-Q4_K_M.gguf",
    cache_dir="/kaggle/working/models"
)
print(f"✅ Model: {model_path}")

# Kill any existing server on port 8080
print("\n🔧 Cleaning up any existing server...")
os.system("pkill -9 -f 'llama-server' 2>/dev/null || true")
time.sleep(2)

# ============================================================================
# CRITICAL FIX: Remove the stub directory from LD_LIBRARY_PATH
# The stub was only needed for LINKING. At RUNTIME, we need the REAL CUDA driver!
# ============================================================================
print("\n🔧 Cleaning runtime environment (removing stub paths)...")
STUBS_DIR = "/kaggle/working/cuda_stubs"

# Remove stub directory from LD_LIBRARY_PATH if present
ld_path = os.environ.get("LD_LIBRARY_PATH", "")
if STUBS_DIR in ld_path:
    paths = [p for p in ld_path.split(":") if p and STUBS_DIR not in p]
    os.environ["LD_LIBRARY_PATH"] = ":".join(paths)
    print(f"   Removed {STUBS_DIR} from LD_LIBRARY_PATH")

# Also remove from LIBRARY_PATH (not strictly needed for runtime, but clean)
lib_path = os.environ.get("LIBRARY_PATH", "")
if STUBS_DIR in lib_path:
    paths = [p for p in lib_path.split(":") if p and STUBS_DIR not in p]
    os.environ["LIBRARY_PATH"] = ":".join(paths)
    print(f"   Removed {STUBS_DIR} from LIBRARY_PATH")

print(f"   LD_LIBRARY_PATH: {os.environ.get('LD_LIBRARY_PATH', '(not set)')[:100]}...")

# Start server with multi-GPU
print("\nStarting llama-server with dual T4 config...")
server_cmd = [
    f"{PACKAGE_DIR}/bin/llama-server",
    "-m", model_path,
    "-ngl", "99",
    "--tensor-split", "0.5,0.5",
    "--split-mode", "layer",
    "-fa", "on",
    "--host", "127.0.0.1",
    "--port", "8080",
    "-c", "4096"
]

print(f"Command: {' '.join(server_cmd)}")

# Start with stderr SEPARATE so we can read it
server = subprocess.Popen(
    server_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Wait for server with output capture
print("\nWaiting for server to start (checking every 2s)...")
server_ready = False
collected_output = []

for i in range(45):  # 90 seconds total
    # Check if server crashed
    ret = server.poll()
    if ret is not None:
        print(f"\n❌ Server CRASHED with exit code: {ret}")
        # Read all output
        stdout_data = server.stdout.read().decode('utf-8', errors='ignore')
        stderr_data = server.stderr.read().decode('utf-8', errors='ignore')
        print("\n📋 STDOUT:")
        print(stdout_data[-3000:] if len(stdout_data) > 3000 else stdout_data)
        print("\n📋 STDERR:")
        print(stderr_data[-3000:] if len(stderr_data) > 3000 else stderr_data)
        break
    
    # Try health check
    try:
        r = requests.get("http://127.0.0.1:8080/health", timeout=2)
        if r.status_code == 200:
            print(f"\n✅ Server ready in {(i+1)*2}s!")
            server_ready = True
            break
    except requests.exceptions.ConnectionError:
        pass
    except Exception as e:
        print(f"   Check error: {e}")
    
    if i % 5 == 4:
        print(f"   Still waiting... ({(i+1)*2}s)")
    
    time.sleep(2)
else:
    print("\n⚠️ Server startup timeout (90s)")
    print("\n📋 Attempting to read server output...")
    
    # Try to read any available output without blocking
    try:
        # Kill server to release pipes
        server.terminate()
        time.sleep(1)
        stdout_data = server.stdout.read().decode('utf-8', errors='ignore')
        stderr_data = server.stderr.read().decode('utf-8', errors='ignore')
        if stdout_data:
            print("\n📋 STDOUT:")
            print(stdout_data[-2000:] if len(stdout_data) > 2000 else stdout_data)
        if stderr_data:
            print("\n📋 STDERR:")
            print(stderr_data[-2000:] if len(stderr_data) > 2000 else stderr_data)
        if not stdout_data and not stderr_data:
            print("   (No output captured)")
    except Exception as e:
        print(f"   Error reading output: {e}")

# Check GPU usage
print("\n📊 GPU Memory Usage:")
!nvidia-smi --query-gpu=index,memory.used,memory.total --format=csv

# Also check if llama-server is running
print("\n📋 Process check:")
!ps aux | grep llama-server | grep -v grep || echo "   No llama-server process found"

gemma-2-2b-it-Q4_K_M.gguf:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

✅ Model: /kaggle/working/models/models--lmstudio-community--gemma-2-2b-it-GGUF/snapshots/6aa72da804ad76c5dc862867bfba6256de9172c7/gemma-2-2b-it-Q4_K_M.gguf

🔧 Cleaning up any existing server...

🔧 Cleaning runtime environment (removing stub paths)...
   Removed /kaggle/working/cuda_stubs from LIBRARY_PATH
   LD_LIBRARY_PATH: /usr/local/nvidia/lib:/usr/local/nvidia/lib64...

Starting llama-server with dual T4 config...
Command: /kaggle/working/llamatelemetry-v0.1.0-cuda12-kaggle-t4x2/bin/llama-server -m /kaggle/working/models/models--lmstudio-community--gemma-2-2b-it-GGUF/snapshots/6aa72da804ad76c5dc862867bfba6256de9172c7/gemma-2-2b-it-Q4_K_M.gguf -ngl 99 --tensor-split 0.5,0.5 --split-mode layer -fa on --host 127.0.0.1 --port 8080 -c 4096

Waiting for server to start (checking every 2s)...

✅ Server ready in 6s!

📊 GPU Memory Usage:
index, memory.used [MiB], memory.total [MiB]
0, 1131 MiB, 15360 MiB
1, 1859 MiB, 15360 MiB

📋 Process check:
root        7476 54.7  3.6 14166812 1192848 ? 

In [18]:
# Test inference
import requests
import time

print("Testing multi-GPU inference...")
print("="*60)

start = time.time()
response = requests.post(
    "http://127.0.0.1:8080/v1/chat/completions",
    json={
        "messages": [{"role": "user", "content": "Explain quantum computing in 2 sentences."}],
        "max_tokens": 100,
        "temperature": 0.7
    },
    timeout=60
)
elapsed = time.time() - start

if response.status_code == 200:
    result = response.json()
    content = result["choices"][0]["message"]["content"]
    usage = result.get("usage", {})
    
    print(f"✅ Response ({elapsed:.2f}s):")
    print(f"   {content}")
    print(f"\n📊 Tokens: {usage.get('total_tokens', 'N/A')}")
    if usage.get('completion_tokens'):
        tps = usage['completion_tokens'] / elapsed
        print(f"📊 Speed: {tps:.1f} tokens/sec")
else:
    print(f"❌ Error: {response.status_code}")
    print(response.text)

Testing multi-GPU inference...
✅ Response (0.95s):
   Quantum computing harnesses the principles of quantum mechanics to perform calculations in ways classical computers cannot. By leveraging the "quantum bits" or qubits, which can exist in a superposition of states, quantum computers offer immense potential for solving complex problems in fields like medicine, materials science, and artificial intelligence. 


📊 Tokens: 78
📊 Speed: 64.2 tokens/sec


In [19]:
# Cleanup - stop server
print("Stopping server...")
server.terminate()
server.wait()
print("✅ Server stopped")

# Show final GPU state
print("\n📊 Final GPU State:")
!nvidia-smi --query-gpu=index,memory.used,memory.total,utilization.gpu --format=csv

Stopping server...
✅ Server stopped

📊 Final GPU State:
index, memory.used [MiB], memory.total [MiB], utilization.gpu [%]
0, 3 MiB, 15360 MiB, 2 %
1, 3 MiB, 15360 MiB, 2 %


In [20]:
"""
Split-GPU Architecture Demo:
- GPU 0: llama-server (LLM inference)
- GPU 1: RAPIDS/Graphistry (graph simulation)
"""
import os
import subprocess
import time
import requests
import threading

print("="*70)
print("SPLIT-GPU ARCHITECTURE TEST")
print("="*70)

# ============================================================================
# GPU 0: Start llama-server (LLM)
# ============================================================================
print("\n🔧 GPU 0: Starting llama-server...")

# Force llama-server to use GPU 0 only
llama_env = os.environ.copy()
llama_env["CUDA_VISIBLE_DEVICES"] = "0"

server_cmd = [
    f"{PACKAGE_DIR}/bin/llama-server",
    "-m", model_path,
    "-ngl", "99",
    "-fa", "on",
    "--host", "127.0.0.1",
    "--port", "8080",
    "-c", "4096"
]

server = subprocess.Popen(
    server_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    env=llama_env
)

# Wait for server
for i in range(60):
    try:
        r = requests.get("http://127.0.0.1:8080/health", timeout=2)
        if r.status_code == 200:
            print(f"   ✅ llama-server ready on GPU 0 ({i+1}s)")
            break
    except:
        time.sleep(1)
else:
    print("   ⚠️ Server timeout")

# ============================================================================
# GPU 1: RAPIDS/Graphistry graph operations
# ============================================================================
print("\n🔧 GPU 1: Running RAPIDS graph simulation...")

# Force RAPIDS to use GPU 1 only
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import cudf
import cugraph

# Create sample graph data (simulating knowledge graph from LLM)
edges = cudf.DataFrame({
    "src": [0, 1, 2, 3, 4, 0, 1, 2],
    "dst": [1, 2, 3, 4, 0, 2, 3, 4],
    "weight": [1.0, 2.0, 1.5, 0.5, 3.0, 2.5, 1.0, 0.8]
})

# Create cuGraph graph
G = cugraph.Graph()
G.from_cudf_edgelist(edges, source="src", destination="dst", edge_attr="weight")

print(f"   Graph: {G.number_of_vertices()} vertices, {G.number_of_edges()} edges")

# Run PageRank on GPU 1
pagerank = cugraph.pagerank(G)
print(f"   PageRank computed: {len(pagerank)} nodes")
print(f"   Top node: {pagerank.nlargest(1, 'pagerank')['vertex'].values[0]}")

# ============================================================================
# Combined workflow: LLM query → Graph update
# ============================================================================
print("\n🔗 Combined LLM + Graph workflow...")

# Reset CUDA_VISIBLE_DEVICES for requests
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

# Query LLM on GPU 0
response = requests.post(
    "http://127.0.0.1:8080/v1/chat/completions",
    json={
        "messages": [{"role": "user", "content": "List 3 related concepts to 'machine learning'"}],
        "max_tokens": 100
    },
    timeout=30
)

if response.status_code == 200:
    llm_output = response.json()["choices"][0]["message"]["content"]
    print(f"   LLM (GPU 0): {llm_output[:100]}...")
    
    # Simulate adding LLM-derived edges to graph
    new_edges = cudf.DataFrame({
        "src": [5, 5, 5],
        "dst": [0, 1, 2],
        "weight": [1.0, 1.0, 1.0]
    })
    all_edges = cudf.concat([edges, new_edges])
    G2 = cugraph.Graph()
    G2.from_cudf_edgelist(all_edges, source="src", destination="dst", edge_attr="weight")
    print(f"   Graph (GPU 1): Updated to {G2.number_of_vertices()} vertices")

print("\n📊 GPU Memory Usage:")
!nvidia-smi --query-gpu=index,name,memory.used,memory.total --format=csv

# Cleanup
server.terminate()
server.wait()
print("\n✅ Split-GPU test complete!")

SPLIT-GPU ARCHITECTURE TEST

🔧 GPU 0: Starting llama-server...
   ⚠️ Server timeout

🔧 GPU 1: Running RAPIDS graph simulation...
   Graph: 5 vertices, 8 edges
   PageRank computed: 5 nodes


/usr/local/lib/python3.12/dist-packages/cugraph/link_analysis/pagerank.py:232: UserWarning: Pagerank expects the 'store_transposed' flag to be set to 'True' for optimal performance during the graph creation
  warnings.warn(warning_msg, UserWarning)


   Top node: 2

🔗 Combined LLM + Graph workflow...
   LLM (GPU 0): Here are 3 related concepts to machine learning:

1. **Deep learning:**  A subset of machine learnin...
   Graph (GPU 1): Updated to 6 vertices

📊 GPU Memory Usage:
index, name, memory.used [MiB], memory.total [MiB]
0, Tesla T4, 2773 MiB, 15360 MiB
1, Tesla T4, 3 MiB, 15360 MiB

✅ Split-GPU test complete!


## Step 14: Final Summary

In [21]:
import os
os.chdir("/kaggle/working")

print("="*70)
print("🎉 llamatelemetry v0.1.0 BUILD COMPLETE!")
print("="*70)

print(f"\n📦 Distribution Package:")
!ls -lh {PACKAGE_NAME}.tar.gz

print(f"\n📁 Package Contents:")
!ls -la {PACKAGE_NAME}/

print(f"\n🔧 Binaries:")
!ls -lh {PACKAGE_NAME}/bin/ | head -10

print(f"\n📋 Metadata Summary:")
print(f"   Version: {VERSION}")
print(f"   Platform: Kaggle 2× Tesla T4")
print(f"   CUDA: {cuda_version}")
print(f"   Compute: SM 7.5 (Turing)")
print(f"   FlashAttention: ✅ All quants")
print(f"   Multi-GPU: ✅ Native CUDA")

print(f"\n🚀 Next Steps:")
print(f"   1. Download: {PACKAGE_NAME}.tar.gz")
print(f"   2. Extract: tar -xzf {PACKAGE_NAME}.tar.gz")
print(f"   3. Run: ./start-server.sh model.gguf 8080")

print(f"\n📥 Download from Kaggle Output tab")
print(f"   or copy to output: !cp {PACKAGE_NAME}.tar.gz /kaggle/output/")

🎉 llamatelemetry v0.1.0 BUILD COMPLETE!

📦 Distribution Package:
-rw-r--r-- 1 root root 1.4G Feb  2 19:35 llamatelemetry-v0.1.0-cuda12-kaggle-t4x2.tar.gz

📁 Package Contents:
total 28
drwxr-xr-x 5 root root 4096 Feb  2 19:31 .
drwxr-xr-x 7 root root 4096 Feb  2 19:35 ..
drwxr-xr-x 2 root root 4096 Feb  2 19:31 bin
drwxr-xr-x 2 root root 4096 Feb  2 19:31 include
drwxr-xr-x 2 root root 4096 Feb  2 19:31 lib
-rwxr-xr-x 1 root root  505 Feb  2 19:31 quantize.sh
-rwxr-xr-x 1 root root  699 Feb  2 19:31 start-server.sh

🔧 Binaries:
total 2.6G
-rwxr-xr-x 1 root root 234M Feb  2 19:27 llama-bench
-rwxr-xr-x 1 root root 240M Feb  2 19:27 llama-cli
-rwxr-xr-x 1 root root 238M Feb  2 19:27 llama-cvector-generator
-rwxr-xr-x 1 root root 238M Feb  2 19:26 llama-embedding
-rwxr-xr-x 1 root root 238M Feb  2 19:27 llama-export-lora
-rwxr-xr-x 1 root root 683K Feb  2 19:26 llama-gguf
-rwxr-xr-x 1 root root 749K Feb  2 19:26 llama-gguf-hash
-rwxr-xr-x 1 root root 234M Feb  2 19:27 llama-gguf-split
-rwx

In [22]:
# Copy to Kaggle output for download
import shutil

os.makedirs("/kaggle/output", exist_ok=True)
shutil.copy(f"/kaggle/working/{PACKAGE_NAME}.tar.gz", "/kaggle/output/")
shutil.copy(f"/kaggle/working/{PACKAGE_NAME}.tar.gz.sha256", "/kaggle/output/")

print("✅ Package copied to /kaggle/output/ for download")
!ls -lh /kaggle/output/

✅ Package copied to /kaggle/output/ for download
total 1.4G
-rw-r--r-- 1 root root 1.4G Feb  2 19:37 llamatelemetry-v0.1.0-cuda12-kaggle-t4x2.tar.gz
-rw-r--r-- 1 root root  114 Feb  2 19:37 llamatelemetry-v0.1.0-cuda12-kaggle-t4x2.tar.gz.sha256
